# Phase 4, Stage 3d: Pearson's r (continuous correlation)

## What this adds

Every test so far has relied on the 4 time pressure *bins* — useful for categorical comparisons, but the bin boundaries (75%/50%/25%) are somewhat arbitrary cut points on what is actually a continuous variable (`time_remaining_pct`). Pearson's r sidesteps this entirely: for each rating band, we correlate `time_remaining_pct` directly with `capped_cpl` across all moves in that band, with no binning at all.

- **H0:** there is no linear relationship between `time_remaining_pct` and `capped_cpl` (r = 0).
- **H1:** there is a linear relationship (r != 0).

We expect **r to be negative**: less time remaining -> higher CPL. The hypothesis (in its original form) predicts |r| should be larger for lower-rated bands; our KS/Cohen's d results so far suggest it might instead be larger for higher-rated bands, at least for the first stage of time pressure. This is our cleanest single number per band to settle that question.

## Multiple comparisons

5 correlations (one per rating band). For consistency with the chi-squared tests (also 5 comparisons), we apply **Bonferroni-corrected alpha = 0.05 / 5 = 0.01**.

## A note on interpreting r here

`r` measures the strength of a *linear* relationship specifically. Given everything we've seen so far — medians and means moving in different directions, tails behaving very differently from the bulk of the data — the true relationship between time pressure and CPL is unlikely to be purely linear. A small r doesn't necessarily mean "no relationship", it may mean "not a *linear* one". We'll keep this in mind and treat r as one piece of evidence among several, not the final word.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

ALPHA_BONFERRONI = 0.05 / 5  # 0.01, consistent with the chi-squared tests

RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}

df = pd.read_csv('../../data/processed/analysed_moves.csv')
print(f'Loaded {len(df):,} rows')
print(f'Bonferroni-corrected alpha for Pearson correlations: {ALPHA_BONFERRONI}')

In [ ]:
results = []
for rating_band in sorted(RATING_BAND_LABELS):
    cell = df[df['rating_band'] == rating_band]
    r, p = stats.pearsonr(cell['time_remaining_pct'], cell['capped_cpl'])

    results.append({
        'rating_band': rating_band,
        'rating_band_label': RATING_BAND_LABELS[rating_band],
        'n': len(cell),
        'pearson_r': r,
        'r_squared': r ** 2,
        'p_value': p,
        'significant_bonferroni': p < ALPHA_BONFERRONI,
    })

pearson_results = pd.DataFrame(results)
pearson_results.to_csv('../results/pearson_correlation_results.csv', index=False)
print('Saved to ../results/pearson_correlation_results.csv')
pearson_results